In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# Pasta compartilhada fica em 'Shareddrives' ou 'Shared with me'
# Vamos achar onde está
!ls /content/drive/
!ls /content/drive/MyDrive/ | grep pepe
!find /content/drive -name "adapter_model.safetensors" 2>/dev/null

Mounted at /content/drive
MyDrive


In [ ]:
!find /content/drive -name "adapter_model.safetensors" 2>/dev/null
!ls /content/drive/MyDrive/

/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/adapter_model.safetensors
/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/checkpoints/checkpoint-93/adapter_model.safetensors
pepe-ai


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# Caminho real via shortcut
ROOT  = Path('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai')
LORA  = ROOT / 'lora'
GGUF  = ROOT / 'gguf'
CACHE = ROOT / 'cache'
GGUF.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']            = str(CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(CACHE)

print(f"✅ ROOT:  {ROOT}")
print(f"✅ LORA existe: {LORA.exists()}")
print(f"Arquivos no LORA: {list(LORA.iterdir())}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ROOT:  /content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai
✅ LORA existe: True
Arquivos no LORA: [PosixPath('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/README.md'), PosixPath('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/adapter_model.safetensors'), PosixPath('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/adapter_config.json'), PosixPath('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/chat_template.jinja'), PosixPath('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/tokenizer_config.json'), PosixPath('/content/drive/.shortcut-targets-by-id/1mXyg83UGf7aONo4b39gHpoaV4QrB74VF/pepe-ai/lora/tokenizer.json')]


In [ ]:
# ============================================================
# CÉLULA 2 — Instala dependências
# ============================================================
!pip install -q protobuf==3.20.3
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers>=4.40" accelerate bitsandbytes peft
print("✅ Instalado")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-monitoring 2.30.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-connection 1.21.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-translate 3.26.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
googleapis-common-protos 1.74.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
wandb 0.25.1 requires protobuf!=5.28.0,!=5.29.0,<7,>4.21.0, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataproc 5.26.0 requires protobuf<8.0.0,>=4.25.8, b

In [ ]:
# ============================================================
# CÉLULA 3 — Carrega modelo em f16 + LoRA e exporta GGUF
# ============================================================
import os, gc, torch
os.makedirs("/content/tmp", exist_ok=True)
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"RAM disponível: ", end="")
!free -h | grep Mem | awk '{print $7}'

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = str(LORA),
    max_seq_length = 2048,
    dtype          = torch.float16,
    load_in_4bit   = False,       # f16 limpo para merge correto
    cache_dir      = str(CACHE),
)

print("🔄 Exportando GGUF Q4_K_M...")
model.save_pretrained_gguf(
    str(GGUF),
    tokenizer,
    quantization_method = 'q4_k_m',
    temporary_location  = '/content/tmp',
)
print(f"✅ GGUF salvo em: {GGUF}")
!ls -lah "{GGUF}"

VRAM livre: 15.53 GB
RAM disponível: 10Gi
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

# 🤖 Pepê AI — Fine-tuning com QLoRA

Fine-tuning do **Llama 3 8B** com os dados do Pepê usando **Unsloth + QLoRA**.
Todos os arquivos são salvos no **Disco D (Google Drive)**.

> ⚠️ **Requisito:** Runtime → Alterar tipo de ambiente de execução → **GPU T4**


## ⚙️ Etapa 1 — Instalar dependências
> Após rodar esta célula, o runtime reinicia automaticamente. Continue da **Etapa 2** em diante.

In [ ]:
!nvidia-smi
# Fixa protobuf compatível com Unsloth (sem wandb para evitar conflitos)
!pip install -q protobuf==3.20.3
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "datasets>=2.16" "transformers>=4.40" accelerate bitsandbytes trl huggingface_hub
import os
print('✅ Instalação concluída. Reiniciando runtime...')
os.kill(os.getpid(), 9)

## 💾 Etapa 2 — Montar Disco D (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# ============================================================
# DISCO D = Google Drive / MyDrive / pepe-ai
# ============================================================
D           = Path('/content/drive/MyDrive')
ROOT        = D / 'pepe-ai'
DATA        = ROOT / 'data'
CHECKPOINTS = ROOT / 'checkpoints'
LORA        = ROOT / 'lora'
GGUF        = ROOT / 'gguf'
LOGS        = ROOT / 'logs'
CACHE       = ROOT / 'cache'

for p in [ROOT, DATA, CHECKPOINTS, LORA, GGUF, LOGS, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

# Cache do HuggingFace vai para o Drive também
os.environ['HF_HOME']            = str(CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(CACHE)

print('✅ Disco D montado:')
print(f'   ROOT        → {ROOT}')
print(f'   DATA        → {DATA}')
print(f'   CHECKPOINTS → {CHECKPOINTS}')
print(f'   LORA        → {LORA}')
print(f'   GGUF        → {GGUF}')
print(f'   LOGS        → {LOGS}')
print(f'   CACHE       → {CACHE}')

## 📦 Etapa 3 — Carregar dataset do GitHub

In [ ]:
import json, urllib.request
from datasets import Dataset

BASE_URL = 'https://raw.githubusercontent.com/Joao-Matheus-Amorim/pepe-ai/main/training/datasets/'

def load_jsonl_from_url(url):
    with urllib.request.urlopen(url) as r:
        return [json.loads(line) for line in r.read().decode().splitlines() if line.strip()]

train_data = load_jsonl_from_url(BASE_URL + 'pepe_train.jsonl')
val_data   = load_jsonl_from_url(BASE_URL + 'pepe_val.jsonl')

# Salva cópia local no Disco D
with open(DATA / 'pepe_train.jsonl', 'w', encoding='utf-8') as f:
    for row in train_data: f.write(json.dumps(row, ensure_ascii=False) + '\n')
with open(DATA / 'pepe_val.jsonl', 'w', encoding='utf-8') as f:
    for row in val_data: f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'✅ Treino:    {len(train_data)} exemplos  → {DATA}/pepe_train.jsonl')
print(f'✅ Validação: {len(val_data)} exemplos  → {DATA}/pepe_val.jsonl')
print('\n--- Exemplo ---')
for msg in train_data[0]['messages']:
    role    = msg['role'].upper()
    content = msg.get('content') or '[tool_call]'
    print(f'[{role}] {content[:100]}')

## 🧠 Etapa 4 — Carregar Llama 3 8B + QLoRA

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"🧹 VRAM livre antes de carregar: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024  # ✅ Reduzido de 2048 → economiza ~30% de VRAM na T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/llama-3-8b-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
    cache_dir      = str(CACHE),
)

print(f'✅ Modelo carregado: {model.num_parameters()/1e6:.0f}M parâmetros')
print(f'   GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA pronto: {trainable/1e6:.2f}M treináveis ({100*trainable/total:.2f}%)')

## 📝 Etapa 5 — Formatar dataset e treinar

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

def format_messages(examples):
    texts = []
    for msgs in examples['messages']:
        clean = [m for m in msgs
                 if m.get('role') in ('system', 'user', 'assistant')
                 and m.get('content') is not None]
        texts.append(tokenizer.apply_chat_template(
            clean, tokenize=False, add_generation_prompt=False
        ))
    return {'text': texts}

train_hf = Dataset.from_list(train_data).map(format_messages, batched=True)
val_hf   = Dataset.from_list(val_data).map(format_messages, batched=True)
print(f'✅ Dataset formatado: {len(train_hf)} treino / {len(val_hf)} val')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_hf,
    eval_dataset       = val_hf,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 1,      # ✅ Era 2 → menos RAM de CPU
    packing            = True,   # ✅ Era False → mais eficiente
    args = TrainingArguments(
        per_device_train_batch_size = 1,  # ✅ Era 2 → menos VRAM
        gradient_accumulation_steps = 8,  # ✅ Era 4 → batch efetivo igual (1x8=8)
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        eval_strategy               = 'steps',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 100,
        save_total_limit            = 3,
        output_dir                  = str(CHECKPOINTS),
        logging_dir                 = str(LOGS),
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        report_to                   = 'none',
    ),
)

print('🚀 Iniciando treinamento...')
stats = trainer.train()
print(f'\n✅ Concluído em {stats.metrics["train_runtime"]:.0f}s')
print(f'   Loss final: {stats.metrics["train_loss"]:.4f}')


## 💾 Etapa 6 — Salvar LoRA adapter no Disco D

In [ ]:
model.save_pretrained(str(LORA))
tokenizer.save_pretrained(str(LORA))
print(f'✅ LoRA adapter salvo em: {LORA}')

import os
for f in os.listdir(LORA):
    size = os.path.getsize(LORA / f)
    print(f'   {f}: {size/1e6:.2f} MB')

## 🔄 Etapa 7 — Exportar GGUF Q4_K_M direto no Disco D

In [ ]:
print('🔄 Exportando GGUF Q4_K_M para o Disco D...')
print(f'   Destino: {GGUF}')

model.save_pretrained_gguf(
    str(GGUF),
    tokenizer,
    quantization_method='q4_k_m'
)

import os
print('\n✅ Arquivos gerados no Disco D:')
for f in os.listdir(GGUF):
    size = os.path.getsize(GGUF / f) / 1e9
    print(f'   {f}: {size:.2f} GB')

## 🧪 Etapa 8 — Testar o modelo fine-tunado

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PEPE = (
    'Você é Pepê, um agente de IA pessoal inteligente, direto e eficiente. '
    'Você tem acesso a ferramentas de busca web, clima, visão de tela, execução de comandos, '
    'leitura de arquivos e memória persistente. '
    'Responda sempre em português do Brasil, de forma clara e concisa.'
)

def testar(pergunta):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PEPE},
        {'role': 'user',   'content': pergunta},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(
        input_ids      = inputs,
        max_new_tokens = 256,
        temperature    = 0.7,
        do_sample      = True
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split('assistant')[-1].strip()

perguntas = [
    'Quem é você?',
    'Quem criou você?',
    'Qual a stack do pepe-ai?',
    'O que você consegue fazer?',
    'Qual o próximo passo do projeto?',
]

for p in perguntas:
    print(f'\n[USER] {p}')
    print(f'[PEPÊ] {testar(p)}')

## 🚀 Etapa 9 — Integrar com Ollama no seu PC

Após o treino, o `.gguf` estará em:
```
Google Drive / MyDrive / pepe-ai / gguf / pepe-ft-unsloth.Q4_K_M.gguf
```

No **PowerShell do seu PC** (pasta onde o arquivo está):

```powershell
cd 'D:\pepe-ai\gguf'

@'
FROM ./pepe-ft-unsloth.Q4_K_M.gguf
SYSTEM "Você é Pepê, agente pessoal do João Matheus. Direto, brasileiro, competente."
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
'@ | Out-File -Encoding utf8 Modelfile

ollama create pepe-ft -f Modelfile
ollama run pepe-ft "Quem é você?"
```

Depois edite o `.env` do projeto:
```
PEPE_MODEL_PROVIDER=ollama
PEPE_MODEL_NAME=pepe-ft
```
